# Sample creation Notebook
Make 2 samples per observation type for the following four scenario's
- PASS: Job samples with all passes (from STANDARD_STATUS)
- WARN: Job samples with passes and one or more upper or lower warnings triggered
- FAIL: Job samples containing upper or lower failures
- FAIL_IGN: Job samples containing upper or lower failures that were ignored

A job can be found by a unique combination of JOB_CODE, STD_CODE, STD_LOT_CODE and SCHEME_CODE.
naming scheme of samples: ANOMALY_[Sample Type]_x where x is 1 or 2

data to be saved in data\samples

filters from data\raw\ResultSet.csv: 
- BLANK - ANALYTICAL_TYPE == "Blank"
- CONTROL - ANALYTICAL_TYPE == "Standard" and STD_LOT_CODE not like substring "OREAS"
- SRMS - ANALYTICAL_TYPE == "Standard" and STD_LOT_CODE like substring "OREAS"
- DUP - ANALYTICAL_TYPE == "Replicate"
- REP - ANALYTICAL_TYPE == "Duplicate"

## load historic set

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd


df = pd.read_csv("../data/raw/ResultSet.csv")
df.columns = [c.strip().upper() for c in df.columns]

for col in ["NUMERIC_FINAL_VALUE", "INTERNAL_TARGET_VALUE",
            "INTERNAL_MAX_WARNING_VALUE", "INTERNAL_MIN_WARNING_VALUE",
            "INTERNAL_MAX_VALUE", "INTERNAL_MIN_VALUE"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["ANALYSED_DATE"] = pd.to_datetime(df["ANALYSED_DATE"], errors="coerce")

print(f"Loaded {len(df):,} rows")
print("Available ANALYTICAL_TYPE:")
print(df["ANALYTICAL_TYPE"].value_counts().to_string())

In [ ]:
JOB_COLS = ["JOB_CODE", "STD_CODE", "STD_LOT_CODE", "SCHEME_CODE"]
SCENARIOS = ["PASS", "WARN", "FAIL", "FAIL_IGN"]
SAMPLES_DIR = Path("../data/samples")
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

log_entries = []  # (filename, n_rows) for created samples
skipped = []      # (type_name, scenario) for scenarios with no available job


def classify_job(status_series):
    """Classify a job's rows into PASS/WARN/FAIL/FAIL_IGN from a status column."""
    s = status_series.dropna().astype(str)
    if s.empty:
        return None
    has_fail = s.str.contains("Failure") & ~s.str.contains("Ignor")
    has_fail_ign = s.str.contains("Failure") & s.str.contains("Ignor")
    has_warn = s.str.contains("Warning") & ~s.str.contains("Ignor")
    if has_fail.any():
        return "FAIL"
    if has_fail_ign.any():
        return "FAIL_IGN"
    if has_warn.any():
        return "WARN"
    return "PASS"


def build_scenario_jobs(df_filtered, status_col):
    """Return dict scenario -> sorted list of job-key tuples."""
    job_status = (
        df_filtered.groupby(JOB_COLS)[status_col]
        .apply(classify_job)
        .dropna()
    )
    out = {sc: [] for sc in SCENARIOS}
    for job_key, scenario in job_status.items():
        out[scenario].append(job_key)
    for sc in out:
        out[sc] = sorted(out[sc])
    return out


def save_job_samples(df_filtered, type_name, status_col):
    """Save up to 2 job samples per scenario for one observation type."""
    scenario_jobs = build_scenario_jobs(df_filtered, status_col)
    for scenario in SCENARIOS:
        job_keys = scenario_jobs[scenario][:2]
        if not job_keys:
            skipped.append((type_name, scenario))
            continue
        for i, job_key in enumerate(job_keys, start=1):
            mask = pd.Series(True, index=df_filtered.index)
            for col, val in zip(JOB_COLS, job_key):
                mask &= df_filtered[col] == val
            sample_df = df_filtered[mask]
            filename = f"{type_name}_{scenario}_{i}.csv"
            sample_df.to_csv(SAMPLES_DIR / filename, index=False)
            log_entries.append((filename, len(sample_df)))

## Blank Sample 

In [3]:
df_blank = df[df["ANALYTICAL_TYPE"] == "Blank"]
save_job_samples(df_blank, "BLANK", "STANDARD_STATUS")

## Control sample 

In [4]:
df_control = df[
    (df["ANALYTICAL_TYPE"] == "Standard")
    & ~df["STD_LOT_CODE"].str.contains("OREAS", na=False)
]
save_job_samples(df_control, "CONTROL", "STANDARD_STATUS")

## Duplicate sample

In [5]:
df_dup = df[df["ANALYTICAL_TYPE"] == "Duplicate"]
save_job_samples(df_dup, "DUP", "PRECISION_STATUS")

df_rep = df[df["ANALYTICAL_TYPE"] == "Replicate"]
save_job_samples(df_rep, "REP", "PRECISION_STATUS")

## SRMS sample

In [6]:
df_srms = df[
    (df["ANALYTICAL_TYPE"] == "Standard")
    & df["STD_LOT_CODE"].str.contains("OREAS", na=False)
]
save_job_samples(df_srms, "SRMS", "STANDARD_STATUS")

## Generation log

In [7]:
from datetime import datetime

log_path = SAMPLES_DIR / "sample_generation_log.txt"
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
author = "JurriaanvdS <jjvanderstruijk@hotmail.com>"

lines = ["Test sample generation log", f"Generated: {generated_at}", f"By: {author}", ""]
for filename, n_rows in log_entries:
    lines.append(f"{filename}: {n_rows} records")
if skipped:
    lines.append("")
    lines.append("Skipped (no matching jobs found):")
    for type_name, scenario in skipped:
        lines.append(f"{type_name}_{scenario}: skipped")

log_path.write_text("\n".join(lines) + "\n")
print(f"Wrote {len(log_entries)} sample file(s) and log to {log_path}")

Wrote 38 sample file(s) and log to data\samples\sample_generation_log.txt
